In [1]:
# Install dependencies
!pip install kagglehub lightgbm xgboost -q

import pandas as pd
import numpy as np
import sys
import joblib
import matplotlib.pyplot as plt

# Sklearn — tree-based models
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
import lightgbm as lgb
import xgboost as xgb

# Sklearn — evaluation
from sklearn.metrics import (
    accuracy_score, f1_score, precision_score,
    classification_report, confusion_matrix,
    roc_auc_score
)

# PyTorch — LSTM-AE
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

print("All libraries loaded.")

All libraries loaded.


In [2]:
# Clone repo and import preprocessing module
!git clone https://github.com/YKesX/DTX-AI.git 2>/dev/null
sys.path.append('/content/DTX-AI/services/ai')

from preprocessing import load_data, engineer_features, FEATURES

# Load and engineer features
print("Loading data...")
df = load_data()
df = engineer_features(df)

X = df[FEATURES]
y = df['Fault Label']

print(f"Dataset ready: {X.shape}")
print(f"Class distribution:\n{y.value_counts()}")

Loading data...


100%|██████████| 32.5k/32.5k [00:00<00:00, 26.2MB/s]

Extracting files...
Dataset ready: (995, 9)
Class distribution:
Fault Label
0    607
1    300
2     88
Name: count, dtype: int64


In [ ]:
# ══════════════════════════════════════════════════════
# CONFIGURATION 
# ══════════════════════════════════════════════════════

SPLIT_CONFIGS = [
    (0.8, 0.1, 0.1),   # 80% train, 10% val, 10% test
    (0.7, 0.2, 0.1),   # 70% train, 20% val, 10% test
    (0.6, 0.3, 0.1),   # 60% train, 30% val, 10% test
]

N_ESTIMATORS_LIST = [100, 200, 500]   # for RF, LightGBM, XGBoost
EPOCHS_LIST       = [10, 30, 50]      # for LSTM-AE

RANDOM_STATE = 42

# ══════════════════════════════════════════════════════

In [4]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

def prepare_splits(X, y, train_ratio, val_ratio, test_ratio, random_state=42):
    """
    Splits data into train/val/test sets and applies StandardScaler.

    train_ratio + val_ratio + test_ratio must equal 1.0
    Scaler is fit only on train set to prevent data leakage.
    """
 # First split: separate test set
    X_temp, X_test, y_temp, y_test = train_test_split(
        X, y,
        test_size=test_ratio,
        random_state=random_state,
        stratify=y
    )
 # Second split: separate val from remaining
    val_size_adjusted = val_ratio / (train_ratio + val_ratio)
    X_train, X_val, y_train, y_val = train_test_split(
        X_temp, y_temp,
        test_size=val_size_adjusted,
        random_state=random_state,
        stratify=y_temp
    )
# Scale — fit only on train
    scaler = StandardScaler()
    X_train = pd.DataFrame(scaler.fit_transform(X_train), columns=FEATURES)
    X_val   = pd.DataFrame(scaler.transform(X_val),       columns=FEATURES)
    X_test  = pd.DataFrame(scaler.transform(X_test),      columns=FEATURES)

    return X_train, X_val, X_test, y_train, y_val, y_test, scaler

print("prepare_splits() ready.")

prepare_splits() ready.


In [5]:
from sklearn.metrics import (
    accuracy_score, f1_score, precision_score,
    roc_auc_score, confusion_matrix, ConfusionMatrixDisplay
)
def evaluate_model(model, X_val, y_val, model_name, split_config, param):
    """
    Evaluates model on validation set.
    Returns metrics dict and prints results.
    """

    y_pred = model.predict(X_val)
    y_prob = model.predict_proba(X_val)

    acc       = accuracy_score(y_val, y_pred)
    f1        = f1_score(y_val, y_pred, average='macro')
    precision = precision_score(y_val, y_pred, average='macro', zero_division=0)
    auroc     = roc_auc_score(y_val, y_prob, multi_class='ovr', average='macro')

    print(f"\n{'='*50}")
    print(f"Model     : {model_name}")
    print(f"Split     : {split_config}")
    print(f"Param     : {param}")
    print(f"Accuracy  : {acc:.4f}")
    print(f"F1 (macro): {f1:.4f}")
    print(f"Precision : {precision:.4f}")
    print(f"AUROC     : {auroc:.4f}")

    # Confusion matrix
    cm = confusion_matrix(y_val, y_pred)
    disp = ConfusionMatrixDisplay(cm, display_labels=['no_fault', 'bearing_fault', 'overheating'])
    disp.plot(cmap='Blues')
    plt.title(f"{model_name} | {split_config} | param={param}")
    plt.tight_layout()
    plt.show()

    return {
        'model':     model_name,
        'split':     split_config,
        'param':     param,
        'accuracy':  acc,
        'f1':        f1,
        'precision': precision,
        'auroc':     auroc
    }

print("evaluate_model() ready.")


evaluate_model() ready.


In [ ]:
# RANDOM FOREST
from sklearn.ensemble import RandomForestClassifier

rf_results = []

for split in SPLIT_CONFIGS:
    train_r, val_r, test_r = split

    for n in N_ESTIMATORS_LIST:

        # Prepare data
        X_train, X_val, X_test, y_train, y_val, y_test, scaler = prepare_splits(
            X, y, train_r, val_r, test_r
        )
         # Train
        model = RandomForestClassifier(
            n_estimators=n,
            random_state=RANDOM_STATE,
            class_weight='balanced'  # handles class imbalance
        )
        model.fit(X_train, y_train)

        # Evaluate
        result = evaluate_model(
            model, X_val, y_val,
            model_name='RandomForest',
            split_config=f"{int(train_r*100)}/{int(val_r*100)}/{int(test_r*100)}",
            param=f"n_estimators={n}"
        )
        rf_results.append(result)

print("\nRandom Forest done.")

In [ ]:
# LIGHTGBM
import lightgbm as lgb

lgbm_results = []

for split in SPLIT_CONFIGS:
    train_r, val_r, test_r = split

    for n in N_ESTIMATORS_LIST:

        # Prepare data
        X_train, X_val, X_test, y_train, y_val, y_test, scaler = prepare_splits(
            X, y, train_r, val_r, test_r
        )

        # Train
        model = lgb.LGBMClassifier(
            n_estimators=n,
            random_state=RANDOM_STATE,
            class_weight='balanced',  # handles class imbalance
            verbose=-1                # suppress training logs
        )
        model.fit(X_train, y_train)

        # Evaluate
        result = evaluate_model(
            model, X_val, y_val,
            model_name='LightGBM',
            split_config=f"{int(train_r*100)}/{int(val_r*100)}/{int(test_r*100)}",
            param=f"n_estimators={n}"
        )
        lgbm_results.append(result)

print("\nLightGBM done.")

In [ ]:
# XGBOOST
import xgboost as xgb

xgb_results = []

for split in SPLIT_CONFIGS:
    train_r, val_r, test_r = split

    for n in N_ESTIMATORS_LIST:

        # Prepare data
        X_train, X_val, X_test, y_train, y_val, y_test, scaler = prepare_splits(
            X, y, train_r, val_r, test_r
        )

        # Train
        model = xgb.XGBClassifier(
            n_estimators=n,
            random_state=RANDOM_STATE,
            eval_metric='mlogloss',  # multi-class log loss
            verbosity=0              # suppress training logs
        )
        model.fit(X_train, y_train)

        # Evaluate
        result = evaluate_model(
            model, X_val, y_val,
            model_name='XGBoost',
            split_config=f"{int(train_r*100)}/{int(val_r*100)}/{int(test_r*100)}",
            param=f"n_estimators={n}"
        )
        xgb_results.append(result)

print("\nXGBoost done.")

In [ ]:
# LSTM-AE

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

# ── Model Definition ──────────────────────────────────────
class LSTMAutoencoder(nn.Module):
    def __init__(self, input_dim, hidden_dim, latent_dim):
        super(LSTMAutoencoder, self).__init__()

        # Encoder
        self.encoder = nn.LSTM(input_dim, hidden_dim, batch_first=True)
        self.hidden_to_latent = nn.Linear(hidden_dim, latent_dim)

        # Decoder
        self.latent_to_hidden = nn.Linear(latent_dim, hidden_dim)
        self.decoder = nn.LSTM(hidden_dim, input_dim, batch_first=True)

    def forward(self, x):
        # Encode
        _, (hidden, _) = self.encoder(x)
        latent = self.hidden_to_latent(hidden[-1])

        # Decode
        hidden_dec = self.latent_to_hidden(latent).unsqueeze(0)
        x_reconstructed, _ = self.decoder(
            x, (hidden_dec, torch.zeros_like(hidden_dec))
        )
        return x_reconstructed

# ── LSTM-AE Training Loop ─────────────────────────────────
lstmae_results = []

# LSTM-AE config
INPUT_DIM  = len(FEATURES)   # 9
HIDDEN_DIM = 64
LATENT_DIM = 16
LR         = 0.001
THRESHOLD_PERCENTILE = 95    # anomaly threshold

for split in SPLIT_CONFIGS:
    train_r, val_r, test_r = split

    for n_epochs in EPOCHS_LIST:

        # Prepare data
        X_train, X_val, X_test, y_train, y_val, y_test, scaler = prepare_splits(
            X, y, train_r, val_r, test_r
        )

        # Convert to tensors — shape [samples, 1, features]
        X_train_t = torch.tensor(X_train.values, dtype=torch.float32).unsqueeze(1)
        X_val_t   = torch.tensor(X_val.values,   dtype=torch.float32).unsqueeze(1)

        # Train only on normal data (class 0)
        normal_mask = (y_train == 0).values
        X_normal = X_train_t[normal_mask]

        dataset    = TensorDataset(X_normal, X_normal)
        dataloader = DataLoader(dataset, batch_size=32, shuffle=True)

        # Init model
        model     = LSTMAutoencoder(INPUT_DIM, HIDDEN_DIM, LATENT_DIM)
        optimizer = torch.optim.Adam(model.parameters(), lr=LR)
        criterion = nn.MSELoss()

        # Training
        loss_history = []
        model.train()
        for epoch in range(n_epochs):
            epoch_loss = 0
            for X_batch, _ in dataloader:
                optimizer.zero_grad()
                X_recon = model(X_batch)
                loss    = criterion(X_recon, X_batch)
                loss.backward()
                optimizer.step()
                epoch_loss += loss.item()
            avg_loss = epoch_loss / len(dataloader)
            loss_history.append(avg_loss)

        # Plot loss
        plt.figure(figsize=(8, 3))
        plt.plot(loss_history)
        plt.title(f"LSTM-AE Loss | split={int(train_r*100)}/{int(val_r*100)}/{int(test_r*100)} | epochs={n_epochs}")
        plt.xlabel("Epoch")
        plt.ylabel("MSE Loss")
        plt.tight_layout()
        plt.show()

        # Threshold — 95th percentile of normal train reconstruction error
        model.eval()
        with torch.no_grad():
            recon       = model(X_normal)
            errors      = ((recon - X_normal) ** 2).mean(dim=[1, 2]).numpy()
            threshold   = np.percentile(errors, THRESHOLD_PERCENTILE)

        # Evaluate on val set
        with torch.no_grad():
            recon_val  = model(X_val_t)
            val_errors = ((recon_val - X_val_t) ** 2).mean(dim=[1, 2]).numpy()

        y_pred = (val_errors > threshold).astype(int)
        # LSTM-AE binary: 0=normal, 1=anomaly
        y_val_binary = (y_val != 0).astype(int).values

        acc       = accuracy_score(y_val_binary, y_pred)
        f1        = f1_score(y_val_binary, y_pred, average='macro')
        precision = precision_score(y_val_binary, y_pred, average='macro', zero_division=0)

        print(f"\n{'='*50}")
        print(f"Model     : LSTM-AE")
        print(f"Split     : {int(train_r*100)}/{int(val_r*100)}/{int(test_r*100)}")
        print(f"Epochs    : {n_epochs}")
        print(f"Threshold : {threshold:.4f}")
        print(f"Accuracy  : {acc:.4f}")
        print(f"F1 (macro): {f1:.4f}")
        print(f"Precision : {precision:.4f}")

        lstmae_results.append({
            'model':     'LSTM-AE',
            'split':     f"{int(train_r*100)}/{int(val_r*100)}/{int(test_r*100)}",
            'param':     f"epochs={n_epochs}",
            'accuracy':  acc,
            'f1':        f1,
            'precision': precision,
            'auroc':     None   # binary, skip AUROC for now
        })

print("\nLSTM-AE done.")

In [ ]:
# Combine all results
all_results = rf_results + lgbm_results + xgb_results + lstmae_results
results_df  = pd.DataFrame(all_results)

# ── Results Table ─────────────────────────────────────────
print("\n" + "="*70)
print("FULL RESULTS TABLE")
print("="*70)
print(results_df.to_string(index=False))

# ── Best per model ────────────────────────────────────────
print("\n" + "="*70)
print("BEST CONFIGURATION PER MODEL (by F1)")
print("="*70)
best = results_df.groupby('model').apply(lambda x: x.loc[x['f1'].idxmax()]).reset_index(drop=True)
print(best[['model', 'split', 'param', 'accuracy', 'f1', 'precision', 'auroc']].to_string(index=False))

# ── F1 Comparison Bar Chart ───────────────────────────────
plt.figure(figsize=(12, 5))
for model_name in results_df['model'].unique():
    subset = results_df[results_df['model'] == model_name]
    plt.plot(
        range(len(subset)),
        subset['f1'].values,
        marker='o',
        label=model_name
    )
plt.title("F1 Score Comparison Across Configurations")
plt.xlabel("Configuration Index")
plt.ylabel("F1 Score (macro)")
plt.legend()
plt.tight_layout()
plt.show()

# ── Accuracy Comparison Bar Chart ────────────────────────
plt.figure(figsize=(10, 5))
best_pivot = best.set_index('model')['accuracy']
best_pivot.plot(kind='bar', color='steelblue', edgecolor='black')
plt.title("Best Accuracy per Model")
plt.ylabel("Accuracy")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

print("\nDone. Check results above.")

In [ ]:
# ── Find best model across all results ───────────────────
best_row = results_df[results_df['model'] != 'LSTM-AE'].loc[
    results_df[results_df['model'] != 'LSTM-AE']['f1'].idxmax()
]

print(f"Best model : {best_row['model']}")
print(f"Split      : {best_row['split']}")
print(f"Param      : {best_row['param']}")
print(f"F1         : {best_row['f1']:.4f}")
print(f"Accuracy   : {best_row['accuracy']:.4f}")

# Parse split ratios from string
parts     = [int(x)/100 for x in best_row['split'].split('/')]
train_r, val_r, test_r = parts

X_train, X_val, X_test, y_train, y_val, y_test, scaler = prepare_splits(
    X, y, train_r, val_r, test_r
)

# Combine train + val for final training
X_final = pd.concat([X_train, X_val])
y_final = pd.concat([y_train, y_val])

# Get n_estimators from param string
n_final = int(best_row['param'].split('=')[1])

# Select and train best model
if best_row['model'] == 'RandomForest':
    final_model = RandomForestClassifier(
        n_estimators=n_final,
        random_state=RANDOM_STATE,
        class_weight='balanced'
    )
elif best_row['model'] == 'LightGBM':
    final_model = lgb.LGBMClassifier(
        n_estimators=n_final,
        random_state=RANDOM_STATE,
        class_weight='balanced',
        verbose=-1
    )
elif best_row['model'] == 'XGBoost':
    final_model = xgb.XGBClassifier(
        n_estimators=n_final,
        random_state=RANDOM_STATE,
        eval_metric='mlogloss',
        verbosity=0
    )

final_model.fit(X_final, y_final)

# ── Save model and scaler ─────────────────────────────────
joblib.dump(final_model, 'model_best.pkl')
joblib.dump(scaler, 'scaler.pkl')

# ── Final test set evaluation ─────────────────────────────
y_test_pred = final_model.predict(X_test)
print("\nFinal Test Set Performance:")
print(classification_report(
    y_test, y_test_pred,
    target_names=['no_fault', 'bearing_fault', 'overheating']
))

# Save test data
X_test.to_csv('X_test.csv', index=False)
y_test.to_csv('y_test.csv', index=False)

print("\nSaved:")
print("  model_best.pkl")
print("  scaler.pkl")
print("  X_test.csv")
print("  y_test.csv")